In [4]:
# ==========================================================
# Google Colab - Simple RAG with PDF + Pinecone
# - Upload a PDF
# - Choose ONE chunking strategy
# - Choose ONE embedding model
# - Uses GPT-4o Mini for answering
# ==========================================================

# Install dependencies
# !pip -q install langchain langchain-openai langchain-community langchain-text-splitters \
#                  pinecone pymupdf tiktoken

!pip -q install \
langchain \
langchain-openai \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
sentence-transformers \
pinecone \
pymupdf \
tiktoken
# ==========================
# Google Colab Secrets
# ==========================
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

# ==========================
# Imports
# ==========================
import os

from google.colab import files

from pinecone import Pinecone, ServerlessSpec

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)

from langchain_core.prompts import ChatPromptTemplate

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ==========================
# Upload PDF
# ==========================
uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))

loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

# ==========================================================
# CHUNKING STRATEGIES
# Enable ONLY ONE
# ==========================================================

# --------- Strategy 1 (Enabled) ----------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# --------- Strategy 2 ----------
# text_splitter = CharacterTextSplitter(
#     separator="\n",
#     chunk_size=1000,
#     chunk_overlap=200
# )

# --------- Strategy 3 ----------
# text_splitter = TokenTextSplitter(
#     chunk_size=500,
#     chunk_overlap=100
# )

# --------- Strategy 4 ----------
# from langchain_text_splitters import RecursiveJsonSplitter
# text_splitter = RecursiveJsonSplitter(max_chunk_size=1000)

# --------- Strategy 5 ----------
# from langchain_experimental.text_splitter import SemanticChunker
# text_splitter = SemanticChunker(embeddings)

docs = text_splitter.split_documents(documents)

print(f"Chunks created: {len(docs)}")

# ==========================================================
# EMBEDDING MODELS
# Enable ONLY ONE
# ==========================================================

# --------- OpenAI text-embedding-3-small (Enabled) ----------
# embeddings = OpenAIEmbeddings(
#     model="text-embedding-3-small"
# )

# --------- OpenAI text-embedding-3-large ----------
# embeddings = OpenAIEmbeddings(
#     model="text-embedding-3-large"
# )

# --------- BAAI BGE Small ----------
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-small-en-v1.5"
# )

# --------- BAAI BGE Base ----------
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-base-en-v1.5"
# )

# --------- BAAI BGE Large ----------
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-large-en-v1.5"
# )

# --------- all-MiniLM-L6-v2 ----------
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# --------- all-MiniLM-L12-v2 ----------
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L12-v2"
# )

# ==========================================================
# Pinecone
# ==========================================================

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "rag-demo"

if index_name not in pc.list_indexes().names():
    dimension = len(embeddings.embed_query("hello"))

    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

# ==========================================================
# Create Embeddings & Upload
# ==========================================================

vectors = []

for i, doc in enumerate(docs):
    vector = embeddings.embed_query(doc.page_content)

    vectors.append({
        "id": f"doc-{i}",
        "values": vector,
        "metadata": {
            "text": doc.page_content
        }
    })

index.upsert(vectors=vectors)

print("Documents indexed successfully!")

# ==========================================================
# GPT-4o Mini
# ==========================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# ==========================================================
# Ask Questions
# ==========================================================

query = input("Ask a question: ")

query_embedding = embeddings.embed_query(query)

results = index.query(
    vector=query_embedding,
    top_k=4,
    include_metadata=True
)

context = "\n\n".join(
    match["metadata"]["text"]
    for match in results["matches"]
)

prompt = ChatPromptTemplate.from_template(
    """
Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""
)

response = llm.invoke(
    prompt.format(
        context=context,
        question=query
    )
)

print("\nAnswer:\n")
print(response.content)

Saving AI_ML_Training_Day1_Day2.pdf to AI_ML_Training_Day1_Day2.pdf
Chunks created: 60


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents indexed successfully!
Ask a question: What is the docuemnt about

Answer:

The document is a reference guide for an AI & ML training program focused on unsupervised learning and reinforcement learning. It outlines key concepts, analogies, and examples related to these learning methods, as well as important terms such as training set, underfitting, weight, label/target, learning rate, loss function, model, and overfitting. The guide emphasizes clarity and understanding the rationale behind the techniques before delving into practical applications, which are supported by hands-on labs using Jupyter Notebooks.
